# 图片链接格式转换工具

将 `.qmd` / `.md` 文件中的 HTML `<img>` 标签统一转换为 Markdown `![]()` 格式。

**适用路径：** `D:\github_lianxh\marp-book\` 下的 `chapters\` 和 `appendix\` 文件夹

**转换规则：**
- `<img src="url" width="700" alt="说明">` → `![说明](url){width="700px"}`
- `style="width:60%"` 中的百分比宽度也会被提取
- 没有 width 属性的图片不加尺寸限定
- 代码块（`` ``` `` 包裹的内容）中的 `<img>` 不会被替换

In [1]:
import re
import os
import glob
from pathlib import Path

In [2]:
# ── 配置区 ──────────────────────────────────────────────
#ROOT = r"D:\github_lianxh\marp-book"
ROOT = os.getcwd()
SCAN_DIRS = ["chapters", "appendix"]   # 扫描的子文件夹
EXTENSIONS = [".qmd", ".md"]           # 处理的文件类型
DRY_RUN = False   # True = 只预览，不写入；改为 False 才真正替换
# ────────────────────────────────────────────────────────

In [3]:
def extract_attr(attrs: str, name: str) -> str:
    """从 HTML 属性字符串中提取指定属性值"""
    m = re.search(rf'{name}=["\']([^"\']*)["\']', attrs, re.IGNORECASE)
    return m.group(1).strip() if m else ''


def extract_width(attrs: str) -> str:
    """
    提取宽度，支持以下几种写法：
      width="700"  width="700px"  width="60%"
      style="width:700px"  style="width:60%"
    """
    # 优先取 width 属性
    w = extract_attr(attrs, 'width')
    if w:
        # 统一加 px（如果是纯数字）
        return w if (w.endswith('%') or w.endswith('px')) else f'{w}px'

    # 再从 style 里找 width
    style = extract_attr(attrs, 'style')
    if style:
        m = re.search(r'width\s*:\s*([\d.]+(?:px|%))', style, re.IGNORECASE)
        if m:
            return m.group(1)
    return ''


def img_tag_to_markdown(tag: str) -> str:
    """把单个 <img ...> 标签转换为 Markdown 格式"""
    # 去掉标签本身，只留属性字符串
    attrs = re.sub(r'^<img\s*', '', tag, flags=re.IGNORECASE)
    attrs = re.sub(r'\s*/?>$', '', attrs)

    src   = extract_attr(attrs, 'src')
    alt   = extract_attr(attrs, 'alt')
    width = extract_width(attrs)

    if not src:
        return tag  # 没有 src，不动

    width_spec = f'{{width="{width}"}}' if width else ''
    return f'![{alt}]({src}){width_spec}'


IMG_PATTERN = re.compile(r'<img\s[^>]*>', re.IGNORECASE | re.DOTALL)
CODE_BLOCK   = re.compile(r'```.*?```', re.DOTALL)   # 匹配代码块


def convert_file(text: str) -> tuple[str, list[tuple[int, str, str]]]:
    """
    转换文件内容，跳过代码块中的 <img>。
    返回 (转换后的文本, [(行号, 原内容, 新内容), ...])
    """
    # 先标记代码块位置（这些位置的 <img> 不替换）
    code_ranges = [(m.start(), m.end()) for m in CODE_BLOCK.finditer(text)]

    def in_code_block(pos: int) -> bool:
        return any(s <= pos < e for s, e in code_ranges)

    result = []
    last = 0
    diffs = []

    for m in IMG_PATTERN.finditer(text):
        if in_code_block(m.start()):
            result.append(text[last:m.end()])
            last = m.end()
            continue

        original = m.group(0)
        replaced = img_tag_to_markdown(original)

        result.append(text[last:m.start()])
        result.append(replaced)
        last = m.end()

        if original != replaced:
            line_num = text[:m.start()].count('\n') + 1
            diffs.append((line_num, original.strip(), replaced.strip()))

    result.append(text[last:])
    return ''.join(result), diffs

In [4]:
# ── 扫描并转换 ───────────────────────────────────────────
all_files = []
for d in SCAN_DIRS:
    for ext in EXTENSIONS:
        pattern = os.path.join(ROOT, d, '**', f'*{ext}')
        all_files.extend(glob.glob(pattern, recursive=True))

# 也扫描根目录的 index.qmd
for ext in EXTENSIONS:
    root_files = glob.glob(os.path.join(ROOT, f'*{ext}'))
    all_files.extend(root_files)

all_files = sorted(set(all_files))
print(f'找到 {len(all_files)} 个文件\n')

total_changes = 0

for filepath in all_files:
    rel = os.path.relpath(filepath, ROOT)
    original = Path(filepath).read_text(encoding='utf-8')
    converted, diffs = convert_file(original)

    if not diffs:
        continue

    print(f'📄 {rel}  ({len(diffs)} 处)')
    for line_num, old, new in diffs:
        print(f'   行 {line_num}')
        print(f'   - {old[:120]}')
        print(f'   + {new[:120]}')
    print()

    if not DRY_RUN:
        Path(filepath).write_text(converted, encoding='utf-8')

    total_changes += len(diffs)

print('─' * 60)
if DRY_RUN:
    print(f'[预览模式]  共找到 {total_changes} 处可替换的 <img> 标签')
    print('将 DRY_RUN 改为 False 后重新运行，即可写入文件。')
else:
    print(f'[已写入]  共替换 {total_changes} 处 <img> 标签')

找到 17 个文件

📄 chapters\ch00.qmd  (1 处)
   行 9
   - <img src="https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20260320215405.png" width="700" alt="20260320215405">
   + ![20260320215405](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20260320215405.png){width="700px"}

📄 chapters\ch01.qmd  (4 处)
   行 100
   - <img src="https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/Marp%E5%88%9B%E5%BB%BA%E5%B9%BB%E7%81%AF%E7%89%87_Fig6_%20Marp
   + ![](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/Marp%E5%88%9B%E5%BB%BA%E5%B9%BB%E7%81%AF%E7%89%87_Fig6_%20Marp%E6%8F
   行 110
   - <img src="https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20260320213614.png" width="700">
   + ![](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20260320213614.png){width="700px"}
   行 158
   - <img src="https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20260320212434.png" width="500" alt="20260320212434">
   + ![20260320212434](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20260320212434.png){width="500px"}
   行 208
   - 

---
## 使用说明

1. **预览**：保持 `DRY_RUN = True`，运行上方单元格，查看会被替换的内容
2. **执行**：确认无误后，将 `DRY_RUN` 改为 `False`，再次运行即可写入
3. **路径**：如果项目位置不同，修改 `ROOT` 变量即可
4. **范围**：默认扫描 `chapters/`、`appendix/` 及根目录的 `.qmd`/`.md` 文件，可在 `SCAN_DIRS` 和 `EXTENSIONS` 中调整